In [ ]:
!pip install -q -U transformers accelerate fastapi uvicorn pyngrok

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

In [ ]:
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

MODEL_ID = "Qwen/Qwen2.5-Omni-3B"

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Qwen loaded!")

In [ ]:
import threading

# Every task is defined once, here. The API and the helper functions below
# both use this dict, so the task list and the JSON schemas cannot drift apart.
PROMPTS = {
    "lyrics_analysis": """
Analyse these lyrics.

Return ONLY valid JSON with:
themes, emotional_tone, sentiment, genres,
instruments, listening_mood, explanation.

Lyrics: {content}
""",

    "search_understanding": """
Understand this music search query.
Extract the user's intent and preferences.

Return ONLY valid JSON:
{
  "original_query": "",
  "search_query": "",
  "artist": null,
  "genre": [],
  "mood": [],
  "language": null,
  "tempo": null,
  "intent": "",
  "keywords": []
}

Copy the query unchanged into original_query.
Do not invent artists or songs.

Query: {content}
""",

    "voice_query_understanding": """
Interpret this speech-recognized music query.
Correct obvious speech recognition errors where context allows.

Return ONLY valid JSON:
{
  "original_transcript": "",
  "corrected_query": "",
  "intent": "",
  "genre": [],
  "mood": [],
  "artist": null,
  "keywords": []
}

Copy the transcript unchanged into original_transcript.
Do not invent artists or songs.

Transcript: {content}
""",

    "prompt_enhancement": """
Improve this music generation prompt.

Return ONLY valid JSON with:
enhanced_prompt, genre, mood, instruments, tempo.

Preserve the user's original intention.

Request: {content}
""",

    "recommendation_explanation": """
Explain why these music recommendations match the user.

Return ONLY valid JSON with:
explanation, mood_match, genre_match.

Context: {content}
""",

    "discover_personalization": """
Analyse the supplied user preferences and music context.

Return ONLY valid JSON with:
recommended_moods, recommended_genres,
recommended_searches, explanation.

Context: {content}
""",

    "music_analysis": """
Interpret these measured audio-analysis results.

Return ONLY valid JSON:
{
  "overall_description": "",
  "detected_mood": [],
  "musical_character": "",
  "energy_interpretation": "",
  "listening_context": "",
  "explanation": ""
}

Do not change or invent BPM, key, energy or other measurements.
Interpret only the supplied metrics.

Audio metrics: {content}
""",

    "listening_preference_analysis": """
Analyse the user's actual music library.

Return ONLY valid JSON:
{
  "listening_personality": "",
  "preferred_genres": [],
  "preferred_moods": [],
  "listening_patterns": [],
  "recommendation_direction": "",
  "insights": []
}

Base all conclusions on the supplied tracks.
Do not invent listening history or preferences.

Listening data: {content}
""",

    "ai_dj_transition": """
Evaluate a possible transition between two tracks.

Return ONLY valid JSON:
{
  "transition_suitability": "",
  "energy_compatibility": "",
  "tempo_compatibility": "",
  "mood_compatibility": "",
  "transition_suggestion": "",
  "explanation": ""
}

Use only the supplied metadata.
Do not invent BPM or musical properties.

Tracks: {content}
""",
}

ALLOWED_TASKS = list(PROMPTS.keys())

# One GPU model: run one generation at a time even if requests arrive together.
inference_lock = threading.Lock()


def moodmix_qwen(task, content):
    if task not in PROMPTS:
        raise ValueError(f"Unknown task: {task}")

    # .replace (not .format / f-string) so the JSON braces in the prompts
    # and any braces inside the user's content can never break formatting.
    prompt = PROMPTS[task].replace("{content}", str(content))

    conversation = [{
        "role": "user",
        "content": [{"type": "text", "text": prompt}]
    }]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to(model.device)

    with inference_lock, torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            return_audio=False
        )

    input_length = inputs["input_ids"].shape[1]
    generated = output[:, input_length:]

    result = processor.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return result[0]


print("MoodMix Qwen functions ready!")
print("Supported tasks:", ALLOWED_TASKS)

In [ ]:
# Thin wrappers: they only pass the raw data. All prompt text lives in PROMPTS.

def understand_search(query):
    return moodmix_qwen("search_understanding", query)


def understand_voice_query(transcript):
    return moodmix_qwen("voice_query_understanding", transcript)


def interpret_music_analysis(audio_metrics):
    return moodmix_qwen("music_analysis", audio_metrics)


def analyze_library(recent_tracks, liked_tracks):
    content = f"RECENTLY PLAYED:\n{recent_tracks}\n\nLIKED TRACKS:\n{liked_tracks}"
    return moodmix_qwen("listening_preference_analysis", content)


def analyze_dj_transition(track_a, track_b):
    content = f"TRACK A:\n{track_a}\n\nTRACK B:\n{track_b}"
    return moodmix_qwen("ai_dj_transition", content)

In [ ]:
result = moodmix_qwen(
    "lyrics_analysis",
    "Walking through the city lights, thinking about the days gone by."
)

print(result)

In [ ]:
tests = {
    "lyrics_analysis": "I miss the days when we were young and free.",
    "search_understanding": "Find energetic Hindi rock songs for working out.",
    "prompt_enhancement": "Generate calm piano music for a rainy evening.",
    "recommendation_explanation": "The user likes acoustic music, soft vocals and reflective songs."
}

for task, content in tests.items():
    print(f"\n--- {task} ---")
    print(moodmix_qwen(task, content))

In [ ]:
# Authenticate ngrok using Kaggle Secrets
# Secret Name: NGROK_AUTHTOKEN (Configured via Kaggle Notebook -> Add-ons -> Secrets)
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

try:
    ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception as e:
    raise RuntimeError(
        "Failed to retrieve NGROK_AUTHTOKEN from Kaggle Secrets. "
        "Add it under Kaggle Notebook -> Add-ons -> Secrets with label NGROK_AUTHTOKEN. "
        f"Details: {e}"
    )

if not ngrok_token or not str(ngrok_token).strip():
    raise RuntimeError("NGROK_AUTHTOKEN secret is empty. Provide a valid ngrok authentication token in Kaggle Secrets.")

ngrok.set_auth_token(str(ngrok_token).strip())
print("ngrok authenticated.")

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="MoodMix Qwen API")


class AnalyzeRequest(BaseModel):
    task: str
    content: str


@app.get("/")
def health():
    return {
        "status": "online",
        "model": MODEL_ID.split("/")[-1],
        "supported_tasks": ALLOWED_TASKS
    }


@app.post("/analyze")
def analyze(request: AnalyzeRequest):

    if request.task not in ALLOWED_TASKS:
        raise HTTPException(
            status_code=400,
            detail=f"Invalid task. Supported tasks: {ALLOWED_TASKS}"
        )

    if not request.content.strip():
        raise HTTPException(
            status_code=400,
            detail="Content cannot be empty"
        )

    try:
        result = moodmix_qwen(request.task, request.content)

        return {
            "task": request.task,
            "result": result,
            "status": "success"
        }

    except Exception as e:
        print(f"Qwen error [{request.task}]: {str(e)}")

        raise HTTPException(
            status_code=500,
            detail="Qwen inference failed"
        )


def start_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("MoodMix Qwen API started!")
print("Supported tasks:", ALLOWED_TASKS)

In [ ]:
# Open the tunnel once, after the server is running.
public_url = ngrok.connect(8000).public_url
print("YOUR QWEN API:", public_url)
print("TEST ENDPOINT:", public_url + "/docs")